In [ ]:
import os 
import getpass
import pydantic
from dotenv import load_dotenv
from crewai import LLM, Agent, Task, Crew
from chromadb.config import Settings
from crewai.rag.chromadb.config import ChromaDBConfig
from chromadb.config import Settings
from crewai.tools import tool

llm = LLM(model="ollama/llama3.2:1b-instruct-q8_0", base_url="http://localhost:11434")
#llm = LLM(model="groq/openai/gpt-oss-20b", base_url="https://api.groq.com/openai/v1")

In [ ]:
# Setting up common configfor tool

_tool_config = dict(
    llm=dict(
        provider="ollama", # or google, openai, anthropic, llama2, ...
        config=dict(
            model="llama3.2:1b-instruct-q8_0",
            # temperature=0.5,
            # top_p=1,
            # stream=true,
        ),
    ),
    embedder=dict(
        provider="ollama", # or openai, ollama, ...
        config=dict(
            model_name="all-minilm",
            task_type="RETRIEVAL_DOCUMENT",
            # title="Embeddings",
        ),
    ),
)

#
_rag_tool_config = {
    "embedding_model": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "embedder": {
        "provider": "openai",
        "config": {
            "model": "nomic-embed-text",
            "api_key":"",
            "platform_url":"http://localhost:11434/v1"
        },
    },
    "vectordb": {
        "provider": "chromadb",
        "config": {
            "persist_directory":"agentic-ai/chromadb",
            "allow_reset":"true",
            "is_persistent":"true",
            #"settings": Settings(persist_directory="agentic-ai/chromadb", allow_reset=True, is_persistent=True)
        }
    },
}
   

In [ ]:
from crewai_tools import CodeDocsSearchTool

@tool
def codedoc_search_tool(doc_path: str):
    """
    read api documentation and return content.
    """
    # by providing its URL:
    codedoc_search_tool = CodeDocsSearchTool(
        docs_url='https://docs.oracle.com/javase/8/docs/api/java/net/URL.html',
        config=_tool_config
    )

    #
    return codedoc_search_tool.run()

In [ ]:
import os
from crewai_tools import GithubSearchTool

#@tool
def github_search_tool(github_repo: str):
	"""
    read git hub repo and return repo, code, issues and pr.
    """
    # Initialize the tool for semantic searches within a specific GitHub repository
	_repo = 'https://github.com/brijeshdhaker'
	if github_repo :
		_repo = github_repo
	
	_git_hub_token = os.getenv("GIT_HUB_TOKEN")

	github_search_tool = GithubSearchTool(
		config=_rag_tool_config,
		#github_repo=_repo,
		gh_token=_git_hub_token,
		# Options: code, repo, pr, issue
		content_types=['repo'] 
	)

	return github_search_tool.run("docker-hadoop-cluster")

g_result = github_search_tool(github_repo='https://github.com/brijeshdhaker')
g_result

In [ ]:
from crewai_tools import DOCXSearchTool
# Instantiate Web Search Tool

#@tool
def docx_search_tool(query: str, doc_path :str):
    """
    read workd document and return document content.
    """
	# Initialize the tool for semantic searches within a specific GitHub repository
    _doc_path = '/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx'
    if doc_path :
        _doc_path = doc_path

    # Initialize the tool to search within any DOCX file's content
    docx_tool = DOCXSearchTool(docx=_doc_path, config=_rag_tool_config)
    #
    return docx_tool.run(query)


#
docx_results = docx_search_tool(query="Cloudera", doc_path="/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/docx/Brijesh_Dhaker_ATS_Resume.docx")
docx_results

In [ ]:
from crewai.tools import tool
from crewai_tools import WebsiteSearchTool

# Instantiate Web Search Tool
@tool
def web_search_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = WebsiteSearchTool(
        website='https://example.com',
        config=_tool_config
    )
    #
    return search_tool.run(query)


In [ ]:
from crewai_tools import ScrapeWebsiteTool


# Instantiate Web Search Tool
@tool
def scrap_website_tool(query: str):
    """
    scrap the content of the specified website.
    """
    # Initialize the tool with the website URL, 
    # so the agent can only scrap the content of the specified website
    scrap_website_tool = ScrapeWebsiteTool(
        website_url='https://example.com',
        config=_tool_config
    )
    
    # Extract the text from the site
    #text = scrap_website_tool.run()
    #print(text)

    #
    return scrap_website_tool.run(query)

In [ ]:
# 1. Initialize the tool
from crewai_tools import PDFSearchTool
# - embedding_model (required): choose provider + provider-specific config
# - vectordb (required): choose vector DB and pass its config
@tool
def pdf_search_tool(query : str):
        """_summary
            Searches the pdf documents and returns results.
        """
        pdf_tool = PDFSearchTool(
            pdf='/home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docs/pdf/2026-04-01_BRIJESHD_PROFILE_MP.pdf',
            config={
                "embedding_model": {
                    "provider": "openai",
                    "config": {
                        "model": "nomic-embed-text",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "embedder": {
                    "provider": "openai",
                    "config": {
                        "model": "all-minilm",
                        "api_key":"",
                        "platform_url":"http://localhost:11434/v1"
                    },
                },
                "vectordb": {
                    "provider": "chromadb",  # or "qdrant"
                    "config": {
                        "persist_directory":"agentic-ai/chromadb",
                        "allow_reset":"true",
                        "is_persistent":"true",
                    }
                },
            }
        )

        return pdf_tool.run(query)

#
results = pdf_search_tool.run("Cloudera")
results

In [ ]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# Website Data Ingestion 
loader = UnstructuredURLLoader(urls=["https://docs.crewai.com/how-to/Installing-CrewAI/"])
data = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
texts = text_splitter.split_documents(data)
texts

##### Embaddings

In [ ]:
from com.example.agentic.loader.LoadManager import LoadManager
from com.example.agentic.embedding.EmbeddingManager import EmbeddingManager
from com.example.agentic.vectors.VectorStoreManager import VectorStoreManager
from com.example.agentic.splitter.SplitManager import SplitManager

chunks=[doc.page_content for doc in texts]

### Convert the text to embeddings
embedding_manager=EmbeddingManager()
embeddings = embedding_manager.embed_documents(chunks)

##store int the vector dtaabase
vectorstore = VectorStoreManager()
vectorstore.add_documents(chunks,embeddings)

#### Formatters

In [ ]:
from crewai import Agent, Crew, Process, Task
from crewai.project import CrewBase, agent, crew, task
from crewai_tools import SerperDevTool
from pydantic import BaseModel, Field
from typing import List, Dict
from datetime import datetime

class ResearchPoint(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key findings or insights about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for each finding",
        default_factory=list
    )

class ResearchOutput(BaseModel):
    research_points: List[ResearchPoint] = Field(description="List of research findings")
    summary: str = Field(description="Brief summary of overall findings")

class ResearchImage(BaseModel):
    topic: str = Field(description="The main topic or area being discussed")
    findings: str = Field(description="The key images or design about this topic")
    relevance: str = Field(description="Why this finding is relevant or important")
    sources: List[Dict[str, str]] = Field(
        description="Image with title and url for each finding",
        default_factory=list
    )

class ResearchImageOutput(BaseModel):
    research_images: List[ResearchImage] = Field(description="List of top images on topic")
    summary: str = Field(description="Brief summary of image findings")    

class ExecutiveReportSection(BaseModel):
    section_emoji: str = Field(description="Section emoji (e.g., 🔍, 📊, 🎯)")
    section_title: str = Field(description="Section title")
    section_content: str = Field(description="Main content of the section")
    key_insights: List[str] = Field(description="Key insights from this section")
    recommendations: List[str] = Field(
        default_factory=list,
        description="Optional recommendations based on findings"
    )
    sources: List[Dict[str, str]] = Field(
        description="Sources with title and URL for this section",
        default_factory=list
    )

class ExecutiveReport(BaseModel):
    report_title: str = Field(description="Title of the report")
    generation_date: str = Field(description="Report generation date")
    executive_summary: str = Field(description="A concise executive summary")
    key_findings: List[Dict[str, str]] = Field(
        description="List of key findings with their sources",
        default_factory=list
    )
    report_sections: List[ExecutiveReportSection] = Field(
        description="Detailed report sections"
    )
    next_steps: List[str] = Field(description="Recommended next steps")
    sources: List[Dict[str, str]] = Field(
        description="All sources used in the report",
        default_factory=list
    )

#### Crew Base

In [ ]:
from crewai_tools import ScrapeWebsiteTool
from crewai_tools import TavilySearchTool

from crewai.tools import tool
# Perform search for provide topic
websearch_tool = SerperDevTool()

# 1. Initialize the tool with image search enabled
tavily_tool = TavilySearchTool(
    api_key=os.environ.get("TAVILY_API_KEY"),
    include_images=True,
    max_results=10
)
# Initialize the tool to scrape the target website
@tool
def image_scrap_tool():
    """
    scrape images and content for target website
    """
    #website_url='https://www.designdocs.dev/'
    scrape_tool = ScrapeWebsiteTool()

In [ ]:
from crewai_tools import ScrapeWebsiteTool, SerperDevTool


@CrewBase
class LatestDesignDevelopmentCrew():
    """LatestAiDevelopment crew"""

    agents_config = "agentic-ai/crewai/config/agents.yaml"
    tasks_config = "agentic-ai/crewai/config/tasks.yaml"

    @agent
    def researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['researcher'],
            verbose=True,
            tools=[SerperDevTool()],
            llm=llm
        )
    
    @agent
    def image_researcher(self) -> Agent:
        return Agent(
            config=self.agents_config['image_researcher'],
            verbose=True,
            tools=[tavily_tool],
            allow_delegation=False,
            llm=llm
        )
    # @agent
    # def reporting_analyst(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['reporting_analyst'],
    #         verbose=True
    #     )
    # @agent
    # def formatter(self) -> Agent:
    #     return Agent(
    #         config=self.agents_config['formatter'],
    #         verbose=True
    #     )

    @task
    def research_task(self) -> Task:
        return Task(
            config=self.tasks_config['research_task'],
            output_json=ResearchOutput
        )
    
    @task
    def find_images_task(self) -> Task:
        return Task(
            config=self.tasks_config['find_images_task'],
            output_json=ResearchImageOutput
        )
    
    # #@task
    # def reporting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['reporting_task'],
    #         output_pydantic=ExecutiveReport
    #     )

    # #@task
    # def formatting_task(self) -> Task:
    #     return Task(
    #         config=self.tasks_config['formatting_task']
    #     )
		
    @crew
    def crew(self) -> Crew:
        """Creates the LatestDesignDevelopmentCrew crew"""
        return Crew(
            agents=self.agents, # Automatically created by the @agent decorator
            tasks=self.tasks, # Automatically created by the @task decorator
            process=Process.sequential,
            verbose=True,
        )

In [54]:
#!/usr/bin/env python
import sys
from datetime import datetime

def run():
    """
    Run the crew.
    """
    inputs = {
        'topic': 'Microservice Design',
        'date': datetime.now().strftime('%Y-%m-%d')
    }
    LatestDesignDevelopmentCrew().crew().kickoff(inputs=inputs)

In [ ]:
run()

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9bfb60f2-9ea9-4c37-80a2-5ee7353db8e4                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: research_task                                                                                            │
│  ID: dcc11eb8-e942-4d07-bc8e-5a842999a20d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Task: Conduct a thorough research about Microservice Design Make sure you find any interesting and relevant    │
│  information about Microservice Design.                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_points=[ResearchPoint(topic='Microservice Design', findings='There are several key takeaways from     │
│  the research on microservice design, including its potential to improve scalability and flexibility.',         │
│  relevance='The findings highlight the importance of considering the entire service ecosystem when designing    │
│  microservices.', sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required':       │
│  '[]'}]), ResearchPoint(topic='Microservice Design Best Practices', findings='Several best practices for        │
│  microservice design include using microservices for smaller, independent workloads.', relevance='The findings  │
│  emphasize the importance of tailoring designs to specific use cases and industry requirements.',               │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': '[]'}]),               │
│  ResearchPoint(topic='Microservice Design Architecture', findings='The research highlights the importance of    │
│  adopting a layered architecture for microservices to improve scalability and fault tolerance.',                │
│  relevance='The findings discuss the potential benefits of using service discovery mechanisms and monitoring    │
│  systems to track performance.', sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}',    │
│  'required': '[]'}]), ResearchPoint(topic='Scalability in Microservices', findings='The research emphasizes     │
│  the importance of designing microservices to scale horizontally and vertically.', relevance='The findings      │
│  discuss the potential benefits of using containerization and orchestration tools to improve scalability.',     │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': '[]'}]),               │
│  ResearchPoint(topic='Microservice Design Challenges', findings='The research highlights several challenges     │
│  associated with microservice design, including ensuring interoperability and fault tolerance.',                │
│  relevance='The findings discuss the potential benefits of using microservices to improve communication         │
│  between services.', sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required':    │
│  '[]'}]), ResearchPoint(topic='Service Discovery Mechanisms', findings='The research highlights the importance  │
│  of using service discovery mechanisms to improve horizontal and vertical scaling.', relevance='The findings    │
│  emphasize the potential benefits of using load balancing and circuit breaking algorithms to improve            │
│  performance.', sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required':         │
│  '[]'}]), ResearchPoint(topic='Monitoring Microservices', findings='The research emphasizes the importance of   │
│  using monitoring systems to track performance and identify bottlenecks.', relevance='The findings discuss the  │
│  potential benefits of using Prometheus and Grafana for data collection and visualization.',                    │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': '[]'}]),               │
│  ResearchPoint(topic='Fault Tolerance in Microservices', findings='The research highlights the importance of    │
│  designing microservices to be fault-tolerant and resil

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: research_task                                                                                            │
│  Agent: Microservice Design Senior Solution Architech                                                           │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Design Image Researcher                                                             │
│                                                                                                                 │
│  Task: Review the context you got and expands each topic into full section for a report about Microservice      │
│  Design Make sure you find top 10 interesting and relevant design url about Microservice Design and return      │
│  list of url.                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Microservice Design Design Image Researcher                                                             │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  research_images=[ResearchImage(topic='Scalability in Microservices', findings='The Tavily Search API can be    │
│  used to improve scalability and flexibility by providing real-time search results.', relevance='High',         │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': ''}]),                 │
│  ResearchImage(topic='Fault Tolerance in Microservices', findings='Microservices can be designed to be          │
│  fault-tolerant and resilient by using circuit breaking and service discovery mechanisms.', relevance='High',   │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': ''}]),                 │
│  ResearchImage(topic='Monitoring Microservices', findings='The Tavily Search API can be used to track           │
│  performance and identify bottlenecks using Prometheus and Grafana.', relevance='Medium',                       │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': ''}]),                 │
│  ResearchImage(topic='Service Discovery Mechanisms', findings='The Tavily Search API uses service discovery     │
│  mechanisms such as load balancing and circuit breaking algorithms to improve horizontal and vertical           │
│  scaling.', relevance='High', sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}',       │
│  'required': ''}]), ResearchImage(topic='Microservice Integration with Other Services', findings='The Tavily    │
│  Search API can be used to integrate microservices with other services by using Service Mesh technologies such  │
│  as Istio and Linkerd.', relevance='High', sources=[{'additionalProperties': '', 'type': 'object',              │
│  'properties': '{}', 'required': ''}]), ResearchImage(topic='Microservice Design Patterns',                     │
│  findings='Event-driven architectures and modular designs are common microservice design patterns that help     │
│  improve scalability and maintainability.', relevance='Medium', sources=[{'additionalProperties': '', 'type':   │
│  'object', 'properties': '{}', 'required': ''}]), ResearchImage(topic='Microservice Security', findings='While  │
│  the Tavily Search API is secure by default, encryption and authentication mechanisms must be used to ensure    │
│  strong security.', relevance='Medium', sources=[{'additionalProperties': '', 'type': 'object', 'properties':   │
│  '{}', 'required': ''}]), ResearchImage(topic='Microservice Architecture', findings='A layered architecture is  │
│  recommended for microservices design to improve scalability and fault tolerance.', relevance='Medium',         │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': ''}]),                 │
│  ResearchImage(topic='Microservice Performance', findings='Prometheus and Grafana can be used to track          │
│  performance and identify bottlenecks using data collection and visualization tools.', relevance='Medium',      │
│  sources=[{'additionalProperties': '', 'type': 'object', 'properties': '{}', 'required': ''}]),                 │
│  ResearchImage(topic='Microservice Scalability', findings='Containerization and orchestration tools such as     │
│  Kubernetes can be used to improve scalability on AWS.', relevance='High', sources=[{'additionalProperties':    │
│  '', 'type': 'object', 'properties': '{}', 'required': 

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: find_images_task                                                                                         │
│  Agent: Microservice Design Design Image Researcher                                                             │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9bfb60f2-9ea9-4c37-80a2-5ee7353db8e4                                                                       │
│  Final Output: {"research_images":[{"topic":"Scalability in Microservices","findings":"The Tavily Search API    │
│  can be used to improve scalability and flexibility by providing real-time search                               │
│  results.","relevance":"High","sources":[{"additionalProperties":"","type":"object","properties":"{}","require  │
│  d":""}]},{"topic":"Fault Tolerance in Microservices","findings":"Microservices can be designed to be           │
│  fault-tolerant and resilient by using circuit breaking and service discovery                                   │
│  mechanisms.","relevance":"High","sources":[{"additionalProperties":"","type":"object","properties":"{}","requ  │
│  ired":""}]},{"topic":"Monitoring Microservices","findings":"The Tavily Search API can be used to track         │
│  performance and identify bottlenecks using Prometheus and                                                      │
│  Grafana.","relevance":"Medium","sources":[{"additionalProperties":"","type":"object","properties":"{}","requi  │
│  red":""}]},{"topic":"Service Discovery Mechanisms","findings":"The Tavily Search API uses service discovery    │
│  mechanisms such as load balancing and circuit breaking algorithms to improve horizontal and vertical           │
│  scaling.","relevance":"High","sources":[{"additionalProperties":"","type":"object","properties":"{}","require  │
│  d":""}]},{"topic":"Microservice Integration with Other Services","findings":"The Tavily Search API can be      │
│  used to integrate microservices with other services by using Service Mesh technologies such as Istio and       │
│  Linkerd.","relevance":"High","sources":[{"additionalProperties":"","type":"object","properties":"{}","require  │
│  d":""}]},{"topic":"Microservice Design Patterns","findings":"Event-driven architectures and modular designs    │
│  are common microservice design patterns that help improve scalability and                                      │
│  maintainability.","relevance":"Medium","sources":[{"additionalProperties":"","type":"object","properties":"{}  │
│  ","required":""}]},{"topic":"Microservice Security","findings":"While the Tavily Search API is secure by       │
│  default, encryption and authentication mechanisms must be used to ensure strong                                │
│  security.","relevance":"Medium","sources":[{"additionalProperties":"","type":"object","properties":"{}","requ  │
│  ired":""}]},{"topic":"Microservice Architecture","findings":"A layered architecture is recommended for         │
│  microservices design to improve scalability and fault                                                          │
│  tolerance.","relevance":"Medium","sources":[{"additionalProperties":"","type":"object","properties":"{}","req  │
│  uired":""}]},{"topic":"Microservice Performance","findings":"Prometheus and Grafana can be used to track       │
│  performance and identify bottlenecks using data collection and visualization                                   │
│  tools.","relevance":"Medium","sources":[{"additionalProperties":"","type":"object","properties":"{}","require  │
│  d":""}]},{"topic":"Microservice Scalability","findings":"Containerization and orchestration tools such as      │
│  Kubernetes can be used to improve scalability on                                                               │
│  AWS.","relevance":"High","sources":[{"additionalPrope

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ 0d1bcdbb-495e-4d8d-9e8d-5a4df06239f4                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/0d1bcdbb-495e-4d8 │
│ d-9e8d-5a4df06239f4?access_code=TRACE-3a9173ef72                             │
│ 🔑 Access Code: TRACE-3a9173ef72                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


####
####

In [ ]:
from crewai.tools import tool
from langchain_community.tools import DuckDuckGoSearchResults
from crewai_tools import SerperDevTool

serper_tool = SerperDevTool(
    name="Web Search Tool",
    description="A tool to perform web searches and retrieve search results.",
    verbose=True
)

import json


@tool
def search_web_tool(query: str):
    """
    Searches the web and returns results.
    """
    search_tool = DuckDuckGoSearchResults(num_results=5, verbose=True)
    return search_tool.run(query)


# Initialize the tool to scrape the target website
scrape_tool = ScrapeWebsiteTool(website_url='https://www.designdocs.dev/')

In [ ]:
from crewai_tools import ScrapeWebsiteTool
from crewai import Agent




##### Create Tasks

##### Create Crew

In [ ]:

# Assemble the Crew
from crewai import Crew, Process

crew = Crew(
    agents=[web_researcher, image_researcher, document_writer],
    tasks=[web_search_task, image_search_task,write_task],
    #process=Process.hierarchical,
    manager_llm=llm,
    planning=True,  # Enable AI planning feature
    planning_llm=llm,
    full_output=True,
    share_crew=False,
    verbose=True
)

# Execute the Tasks
crew.kickoff()

In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool, FileWriterTool

# 1. Initialize Tools
search_tool = SerperDevTool()
file_writer = FileWriterTool()

# 2. Define Agents
researcher = Agent(
    role='Researcher', 
    goal='Find image URLs',
    backstory="""A design expert with a passion for sharing the best experiences in microservice desing.""", 
    tools=[search_tool],
    verbose=True,
    max_iter=3, 
    llm=llm,
    allow_delegation=False
)

writer = Agent(
    role='Writer', 
    goal='Create markdown file with images and summary', 
    backstory="""You are a solution architech and design expert with a passion for writing bleog for microservice desing.""", 
    tools=[file_writer], 
    verbose=True,
    max_iter=3, 
    llm=llm,
    allow_delegation=False
)

# 3. Define Tasks
task1 = Task(
    description="Search for images of [microservice design]",
    expected_output='List top 5 images of [microservice design]', 
    agent=researcher
)

task2 = Task(
    description="Save images as a Markdown list in 'agentic-ai/crewai/desings/project-design_images.md'",
    expected_output='A summary of the top 3 microservice desings.', 
    agent=writer, 
    output_file='agentic-ai/crewai/desings/project-design_images.md'
)

# 4. Form and Run Crew
crew = Crew(agents=[researcher, writer], tasks=[task1, task2])
crew.kickoff()